In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, f1_score
from sklearn.tree import DecisionTreeClassifier


In [ ]:
df=pd.read_csv("/content/drive/MyDrive/skyguard_weather.csv")

In [ ]:
df


,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa
0,Central_UP,42189,Bareilly,Bareilly,28.3667,79.4000,2024-01-01 00:00:00,10.6,87.0,1018.9
1,Central_UP,42189,Bareilly,Bareilly,28.3667,79.4000,2024-01-01 01:00:00,11.0,88.0,1018.6
2,Central_UP,42189,Bareilly,Bareilly,28.3667,79.4000,2024-01-01 02:00:00,11.0,88.0,1019.0
3,Central_UP,42189,Bareilly,Bareilly,28.3667,79.4000,2024-01-01 03:00:00,11.2,95.0,1020.2
4,Central_UP,42189,Bareilly,Bareilly,28.3667,79.4000,2024-01-01 04:00:00,13.0,72.0,1021.0
...,...,...,...,...,...,...,...,...,...,...
459404,West_Bengal,42803,Midnapore,Midnapore,22.4167,87.3167,2026-08-24 06:00:00,27.2,89.0,1001.3
459405,West_Bengal,42803,Midnapore,Midnapore,22.4167,87.3167,2026-08-24 07:00:00,27.4,92.0,1000.3
459406,West_Bengal,42803,Midnapore,Midnapore,22.4167,87.3167,2026-08-24 08:00:00,30.2,83.0,998.9
459407,West_Bengal,42803,Midnapore,Midnapore,22.4167,87.3167,2026-08-24 09:00:00,30.4,81.0,998.3


In [ ]:
df.shape

(459409, 10)

In [ ]:
df.isna().sum()

,0
cluster,0
station_id,0
station_name,0
city,0
latitude,0
longitude,0
timestamp,0
temperature_c,0
humidity_pct,5
pressure_hpa,19


In [ ]:
df = df.sort_values(by=['station_id', 'timestamp']).reset_index(drop=True) #sorting as per time and station
df[['humidity_pct', 'pressure_hpa']] = df.groupby('station_id')[['humidity_pct', 'pressure_hpa']].transform(lambda group: group.interpolate(method='linear').ffill().bfill())

In [ ]:
df

,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa
0,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 00:00:00,5.9,100.0,1018.6
1,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 01:00:00,5.6,100.0,1019.1
2,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 02:00:00,6.6,99.0,1019.8
3,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 03:00:00,10.4,84.0,1020.5
4,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 04:00:00,8.5,98.0,1020.8
...,...,...,...,...,...,...,...,...,...,...
459404,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 06:00:00,31.6,65.0,1008.9
459405,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 07:00:00,32.0,63.0,1008.2
459406,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 08:00:00,32.6,61.0,1006.9
459407,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 09:00:00,32.5,63.0,1005.6


In [ ]:
df.isna().sum()

,0
cluster,0
station_id,0
station_name,0
city,0
latitude,0
longitude,0
timestamp,0
temperature_c,0
humidity_pct,0
pressure_hpa,0


In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp']) #conversion of time in int form
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0) # in sin and cos form to understand a day , hour , month is complete or not
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

In [ ]:
df

,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa,hour,month,day,hour_sin,hour_cos,month_sin,month_cos
0,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 00:00:00,5.9,100.0,1018.6,0,1,1,0.000000,1.000000e+00,0.500000,0.866025
1,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 01:00:00,5.6,100.0,1019.1,1,1,1,0.258819,9.659258e-01,0.500000,0.866025
2,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 02:00:00,6.6,99.0,1019.8,2,1,1,0.500000,8.660254e-01,0.500000,0.866025
3,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 03:00:00,10.4,84.0,1020.5,3,1,1,0.707107,7.071068e-01,0.500000,0.866025
4,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-01 04:00:00,8.5,98.0,1020.8,4,1,1,0.866025,5.000000e-01,0.500000,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459404,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 06:00:00,31.6,65.0,1008.9,6,8,24,1.000000,6.123234e-17,-0.866025,-0.500000
459405,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 07:00:00,32.0,63.0,1008.2,7,8,24,0.965926,-2.588190e-01,-0.866025,-0.500000
459406,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 08:00:00,32.6,61.0,1006.9,8,8,24,0.866025,-5.000000e-01,-0.866025,-0.500000
459407,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 09:00:00,32.5,63.0,1005.6,9,8,24,0.707107,-7.071068e-01,-0.866025,-0.500000


In [ ]:
df['cluster_temp_mean'] = df.groupby(['cluster', 'timestamp'])['temperature_c'].transform('mean') # doing group by to analyze and preprocess more nicely
df['spatial_temp_diff'] = df['temperature_c'] - df['cluster_temp_mean']
df['cluster_press_mean'] = df.groupby(['cluster', 'timestamp'])['pressure_hpa'].transform('mean')
df['spatial_press_diff'] = df['pressure_hpa'] - df['cluster_press_mean']
df['temp_press_ratio'] = df['temperature_c'] / (df['pressure_hpa'] + 1e-5) #values might be 0 so used 1e-5 to be safe

In [ ]:
target_cols = ['temperature_c', 'humidity_pct', 'pressure_hpa'] #using lag 1hour, 24 hour and roll mean of 6 hour and 24 hour as we need some previous data to explain bout the current situation i
#i.e. some memory is required
#maked it oprimized qith quantile for example there might be anomay just before 1 hour so lag 1 hour will have anomaly so to save from it we wrote upper and lower bound
for col in target_cols:
    lower_bound = df[col].quantile(0.001)
    upper_bound = df[col].quantile(0.999)
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
new_feature_columns = []
for col in target_cols:
    grouped = df.groupby('station_id')[col]
    lag1 = grouped.shift(1).rename(f'{col} lag1')
    lag24 = grouped.shift(24).rename(f'{col} lag24')
    roll6_med = grouped.transform(lambda x: x.rolling(6, min_periods=1).median()).rename(f'{col}_roll6_med')
    roll24_mean = grouped.transform(lambda x: x.rolling(24, min_periods=1).mean()).rename(f'{col}_roll24_mean')
    new_feature_columns.extend([lag1, lag24, roll6_med, roll24_mean])


df_clean = pd.concat([df] + new_feature_columns, axis=1)

In [ ]:
df_clean.dropna(inplace=True)

In [ ]:
df_clean

,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa,...,temperature_c_roll6_med,temperature_c_roll24_mean,humidity_pct lag1,humidity_pct lag24,humidity_pct_roll6_med,humidity_pct_roll24_mean,pressure_hpa lag1,pressure_hpa lag24,pressure_hpa_roll6_med,pressure_hpa_roll24_mean
24,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 00:00:00,6.0,100.0,1018.0,...,6.25,10.129167,100.0,100.0,99.5,87.583333,1017.8,1018.6,1018.05,1018.500000
25,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 01:00:00,6.3,100.0,1018.4,...,6.15,10.141667,100.0,100.0,100.0,87.583333,1018.0,1019.1,1018.05,1018.470833
26,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 02:00:00,6.6,99.0,1019.1,...,6.15,10.141667,100.0,99.0,100.0,87.583333,1018.4,1019.8,1018.05,1018.441667
27,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 03:00:00,9.6,87.0,1020.2,...,6.15,10.108333,99.0,84.0,100.0,87.708333,1019.1,1020.5,1018.20,1018.429167
28,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 04:00:00,8.0,98.0,1020.7,...,6.45,10.087500,87.0,98.0,99.5,87.708333,1020.2,1020.8,1018.75,1018.425000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459404,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 06:00:00,31.6,65.0,1008.9,...,29.35,28.066667,67.0,60.0,74.5,83.375000,1009.4,1008.6,1009.20,1007.962500
459405,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 07:00:00,32.0,63.0,1008.2,...,30.50,28.029167,65.0,61.0,69.0,83.458333,1008.9,1007.9,1009.20,1007.975000
459406,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 08:00:00,32.6,61.0,1006.9,...,31.25,28.020833,63.0,64.0,66.0,83.333333,1008.2,1007.0,1009.15,1007.970833
459407,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 09:00:00,32.5,63.0,1005.6,...,31.80,28.012500,61.0,67.0,64.0,83.166667,1006.9,1005.9,1008.55,1007.958333


In [ ]:
df_injected = df_clean.copy() #making a copy to analyse and compare
df_injected['anomaly_label'] = 0  #0 means Normal and 1 means Anomaly
df_injected['root_cause'] = 'Normal'

In [ ]:
np.random.seed(42)    #injecting some random data to analyze that can our model detect it or not :)
spike_indices = np.random.choice(df_injected.index, size=int(len(df_injected) * 0.01), replace=False)
df_injected.loc[spike_indices, 'temperature_c'] += np.random.uniform(12, 25, size=len(spike_indices))
df_injected.loc[spike_indices, 'anomaly_label'] = 1
df_injected.loc[spike_indices, 'root_cause'] = 'Sudden_Spike'

freeze_starts = np.random.choice(df_injected.index[:-10], size=100, replace=False)
for idx in freeze_starts:
    df_injected.loc[idx:idx+10, 'humidity_pct'] = df_injected.loc[idx, 'humidity_pct']
    df_injected.loc[idx:idx+10, 'anomaly_label'] = 1
    df_injected.loc[idx:idx+10, 'root_cause'] = 'Frozen_Sensor'

In [ ]:
df_clean

,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa,...,temperature_c_roll6_med,temperature_c_roll24_mean,humidity_pct lag1,humidity_pct lag24,humidity_pct_roll6_med,humidity_pct_roll24_mean,pressure_hpa lag1,pressure_hpa lag24,pressure_hpa_roll6_med,pressure_hpa_roll24_mean
24,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 00:00:00,6.0,100.0,1018.0,...,6.25,10.129167,100.0,100.0,99.5,87.583333,1017.8,1018.6,1018.05,1018.500000
25,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 01:00:00,6.3,100.0,1018.4,...,6.15,10.141667,100.0,100.0,100.0,87.583333,1018.0,1019.1,1018.05,1018.470833
26,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 02:00:00,6.6,99.0,1019.1,...,6.15,10.141667,100.0,99.0,100.0,87.583333,1018.4,1019.8,1018.05,1018.441667
27,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 03:00:00,9.6,87.0,1020.2,...,6.15,10.108333,99.0,84.0,100.0,87.708333,1019.1,1020.5,1018.20,1018.429167
28,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 04:00:00,8.0,98.0,1020.7,...,6.45,10.087500,87.0,98.0,99.5,87.708333,1020.2,1020.8,1018.75,1018.425000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459404,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 06:00:00,31.6,65.0,1008.9,...,29.35,28.066667,67.0,60.0,74.5,83.375000,1009.4,1008.6,1009.20,1007.962500
459405,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 07:00:00,32.0,63.0,1008.2,...,30.50,28.029167,65.0,61.0,69.0,83.458333,1008.9,1007.9,1009.20,1007.975000
459406,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 08:00:00,32.6,61.0,1006.9,...,31.25,28.020833,63.0,64.0,66.0,83.333333,1008.2,1007.0,1009.15,1007.970833
459407,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 09:00:00,32.5,63.0,1005.6,...,31.80,28.012500,61.0,67.0,64.0,83.166667,1006.9,1005.9,1008.55,1007.958333


In [ ]:

df_injected

,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa,...,humidity_pct lag1,humidity_pct lag24,humidity_pct_roll6_med,humidity_pct_roll24_mean,pressure_hpa lag1,pressure_hpa lag24,pressure_hpa_roll6_med,pressure_hpa_roll24_mean,anomaly_label,root_cause
24,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 00:00:00,6.0,100.0,1018.0,...,100.0,100.0,99.5,87.583333,1017.8,1018.6,1018.05,1018.500000,0,Normal
25,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 01:00:00,6.3,100.0,1018.4,...,100.0,100.0,100.0,87.583333,1018.0,1019.1,1018.05,1018.470833,0,Normal
26,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 02:00:00,6.6,99.0,1019.1,...,100.0,99.0,100.0,87.583333,1018.4,1019.8,1018.05,1018.441667,0,Normal
27,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 03:00:00,9.6,87.0,1020.2,...,99.0,84.0,100.0,87.708333,1019.1,1020.5,1018.20,1018.429167,0,Normal
28,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 04:00:00,8.0,98.0,1020.7,...,87.0,98.0,99.5,87.708333,1020.2,1020.8,1018.75,1018.425000,0,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459404,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 06:00:00,31.6,65.0,1008.9,...,67.0,60.0,74.5,83.375000,1009.4,1008.6,1009.20,1007.962500,0,Normal
459405,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 07:00:00,32.0,63.0,1008.2,...,65.0,61.0,69.0,83.458333,1008.9,1007.9,1009.20,1007.975000,0,Normal
459406,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 08:00:00,32.6,61.0,1006.9,...,63.0,64.0,66.0,83.333333,1008.2,1007.0,1009.15,1007.970833,0,Normal
459407,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 09:00:00,32.5,63.0,1005.6,...,61.0,67.0,64.0,83.166667,1006.9,1005.9,1008.55,1007.958333,0,Normal


In [ ]:
df_clean.isna().sum()

,0
cluster,0
station_id,0
station_name,0
city,0
latitude,0
longitude,0
timestamp,0
temperature_c,0
humidity_pct,0
pressure_hpa,0


In [ ]:
df_clean = pd.concat([df] + new_feature_columns, axis=1).dropna().reset_index(drop=True) # newfeatures column is made above i.e. lag 1 ,lag 24 etc..

drop_cols = ['station_name', 'city', 'timestamp', 'hour', 'month']
df_clean_model = pd.get_dummies(df_clean.drop(columns=drop_cols), columns=['cluster'], drop_first=True)
bool_cols = df_clean_model.select_dtypes(include='bool').columns
df_clean_model[bool_cols] = df_clean_model[bool_cols].astype(int)

In [ ]:
df_clean_model

,station_id,latitude,longitude,temperature_c,humidity_pct,pressure_hpa,day,hour_sin,hour_cos,month_sin,...,humidity_pct_roll6_med,humidity_pct_roll24_mean,pressure_hpa lag1,pressure_hpa lag24,pressure_hpa_roll6_med,pressure_hpa_roll24_mean,cluster_Konkan_Deccan,cluster_NCR,cluster_Tamil_Nadu_Coast,cluster_West_Bengal
0,42139,29.0167,77.6333,6.0,100.0,1018.0,2,0.000000,1.000000e+00,0.500000,...,99.5,87.583333,1017.8,1018.6,1018.05,1018.500000,0,1,0,0
1,42139,29.0167,77.6333,6.3,100.0,1018.4,2,0.258819,9.659258e-01,0.500000,...,100.0,87.583333,1018.0,1019.1,1018.05,1018.470833,0,1,0,0
2,42139,29.0167,77.6333,6.6,99.0,1019.1,2,0.500000,8.660254e-01,0.500000,...,100.0,87.583333,1018.4,1019.8,1018.05,1018.441667,0,1,0,0
3,42139,29.0167,77.6333,9.6,87.0,1020.2,2,0.707107,7.071068e-01,0.500000,...,100.0,87.708333,1019.1,1020.5,1018.20,1018.429167,0,1,0,0
4,42139,29.0167,77.6333,8.0,98.0,1020.7,2,0.866025,5.000000e-01,0.500000,...,99.5,87.708333,1020.2,1020.8,1018.75,1018.425000,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458924,43331,11.9667,79.8167,31.6,65.0,1008.9,24,1.000000,6.123234e-17,-0.866025,...,74.5,83.375000,1009.4,1008.6,1009.20,1007.962500,0,0,1,0
458925,43331,11.9667,79.8167,32.0,63.0,1008.2,24,0.965926,-2.588190e-01,-0.866025,...,69.0,83.458333,1008.9,1007.9,1009.20,1007.975000,0,0,1,0
458926,43331,11.9667,79.8167,32.6,61.0,1006.9,24,0.866025,-5.000000e-01,-0.866025,...,66.0,83.333333,1008.2,1007.0,1009.15,1007.970833,0,0,1,0
458927,43331,11.9667,79.8167,32.5,63.0,1005.6,24,0.707107,-7.071068e-01,-0.866025,...,64.0,83.166667,1006.9,1005.9,1008.55,1007.958333,0,0,1,0


In [ ]:
df_clean['temp_roll_mean'] = df_clean.groupby('station_id')['temperature_c'].transform(lambda x: x.rolling(24, min_periods=1).mean())
#finding roll mean and making data optimal for operations
df_clean['temp_roll_std'] = (df_clean.groupby('station_id')['temperature_c'].transform(lambda x: x.rolling(24, min_periods=1).std()).fillna(1e-5).replace(0.0, 1e-5))

#safe z score with no NAN values or 0 values
df_clean['stat_zscore'] = ((df_clean['temperature_c'] - df_clean['temp_roll_mean']).abs() / df_clean['temp_roll_std']).fillna(0.0)

#normal score
df_clean['stat_anomaly_score'] = (df_clean['stat_zscore'] / 3.0).clip(upper=1.0)

In [ ]:
df_clean

,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa,...,humidity_pct_roll6_med,humidity_pct_roll24_mean,pressure_hpa lag1,pressure_hpa lag24,pressure_hpa_roll6_med,pressure_hpa_roll24_mean,temp_roll_mean,temp_roll_std,stat_zscore,stat_anomaly_score
0,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 00:00:00,6.0,100.0,1018.0,...,99.5,87.583333,1017.8,1018.6,1018.05,1018.500000,6.000000,0.000010,0.000000,0.000000
1,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 01:00:00,6.3,100.0,1018.4,...,100.0,87.583333,1018.0,1019.1,1018.05,1018.470833,6.150000,0.212132,0.707107,0.235702
2,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 02:00:00,6.6,99.0,1019.1,...,100.0,87.583333,1018.4,1019.8,1018.05,1018.441667,6.300000,0.300000,1.000000,0.333333
3,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 03:00:00,9.6,87.0,1020.2,...,100.0,87.708333,1019.1,1020.5,1018.20,1018.429167,7.125000,1.668083,1.483739,0.494580
4,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 04:00:00,8.0,98.0,1020.7,...,99.5,87.708333,1020.2,1020.8,1018.75,1018.425000,7.300000,1.496663,0.467707,0.155902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458924,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 06:00:00,31.6,65.0,1008.9,...,74.5,83.375000,1009.4,1008.6,1009.20,1007.962500,28.066667,2.571527,1.374021,0.458007
458925,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 07:00:00,32.0,63.0,1008.2,...,69.0,83.458333,1008.9,1007.9,1009.20,1007.975000,28.029167,2.503646,1.586020,0.528673
458926,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 08:00:00,32.6,61.0,1006.9,...,66.0,83.333333,1008.2,1007.0,1009.15,1007.970833,28.020833,2.487356,1.840978,0.613659
458927,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 09:00:00,32.5,63.0,1005.6,...,64.0,83.166667,1006.9,1005.9,1008.55,1007.958333,28.012500,2.471281,1.815860,0.605287


In [ ]:
for col in ['temperature_c', 'humidity_pct', 'pressure_hpa']:
    df_injected[f'{col}_roll6_var'] = df_injected.groupby('station_id')[col].transform(lambda x: x.rolling(6, min_periods=1).var()).fillna(0)
    df_clean[f'{col}_roll6_var'] = df_clean.groupby('station_id')[col].transform(lambda x: x.rolling(6, min_periods=1).var()).fillna(0)


In [ ]:
df_injected['cluster_temp_mean'] = df_injected.groupby(['cluster', 'timestamp'])['temperature_c'].transform('mean')
df_injected['spatial_temp_diff'] = df_injected['temperature_c'] - df_injected['cluster_temp_mean']
drop_cols = ['station_name', 'city', 'timestamp', 'hour', 'month','root_cause', 'severity', 'anomaly_label', 'stat_severity']
df_clean_model = pd.get_dummies(df_clean.drop(columns=drop_cols, errors='ignore'), columns=['cluster'], drop_first=True)
bool_cols = df_clean_model.select_dtypes(include='bool').columns
df_clean_model[bool_cols] = df_clean_model[bool_cols].astype(int)

df_injected_model = pd.get_dummies(df_injected.drop(columns=drop_cols, errors='ignore'), columns=['cluster'], drop_first=True)
bool_cols_inj = df_injected_model.select_dtypes(include='bool').columns
df_injected_model[bool_cols_inj] = df_injected_model[bool_cols_inj].astype(int)

In [ ]:
features = [c for c in df_clean_model.columns if c != 'station_id']

df_injected_model = df_injected_model.reindex(columns=df_clean_model.columns, fill_value=0)


iso_model = IsolationForest(contamination=0.03, random_state=42, n_jobs=-1)
iso_model.fit(df_clean_model[features])
raw_scores = iso_model.decision_function(df_injected_model[features])
ml_scores = 1 - ((raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-9))
temp_roll_mean = df_injected.groupby('station_id')['temperature_c'].transform(lambda x: x.rolling(24, min_periods=1).mean())
temp_roll_std = df_injected.groupby('station_id')['temperature_c'].transform(lambda x: x.rolling(24, min_periods=1).std()).fillna(1e-5)
stat_scores = ((df_injected['temperature_c'] - temp_roll_mean).abs() / temp_roll_std / 3.0).clip(upper=1.0)
ensemble_score = (0.40 * stat_scores) + (0.60 * ml_scores)
best_thresh = 0.5
best_f1 = 0
for thresh in np.arange(0.30, 0.90, 0.05):
    preds = (ensemble_score > thresh).astype(int)
    score = f1_score(df_injected['anomaly_label'], preds)
    if score > best_f1:
        best_f1 = score
        best_thresh = thresh


In [ ]:
final_preds = (ensemble_score > best_thresh).astype(int)
print(classification_report(df_injected['anomaly_label'], final_preds))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00    453252
           1       0.85      0.58      0.69      5677

    accuracy                           0.99    458929
   macro avg       0.92      0.79      0.84    458929
weighted avg       0.99      0.99      0.99    458929



In [ ]:
for col in ['temperature_c', 'humidity_pct', 'pressure_hpa']:
    mean_col = df_injected.groupby('station_id')[col].transform(lambda x: x.rolling(24, min_periods=1).mean())
    std_col = df_injected.groupby('station_id')[col].transform(lambda x: x.rolling(24, min_periods=1).std()).fillna(1e-5)
    std_col = std_col.replace(0, 1e-5)
    df_injected[f'{col}_zscore'] = ((df_injected[col] - mean_col).abs() / (std_col * 3.0)).clip(upper=1.0).fillna(0.0)

stat_composite = (df_injected['temperature_c_zscore'] + df_injected['humidity_pct_zscore'] + df_injected['pressure_hpa_zscore']) / 3.0
df_injected['stat_zscore'] = stat_composite.fillna(0.0)
ml_scores_clean = pd.Series(ml_scores, index=df_injected.index).fillna(0.0)
ensemble_score = (0.40 * stat_composite) + (0.60 * ml_scores)
df_injected['ensemble_score'] = ensemble_score

In [ ]:
anomaly_mask = df_injected['anomaly_label'] == 1
X_cause = df_injected_model.loc[anomaly_mask, features]
y_cause = df_injected.loc[anomaly_mask, 'root_cause']
cause_classifier = DecisionTreeClassifier(max_depth=5, random_state=42)
cause_classifier.fit(X_cause, y_cause)


df_injected['predicted_cause'] = 'Normal'
detected_mask = ensemble_score > best_thresh
df_injected.loc[detected_mask, 'predicted_cause'] = cause_classifier.predict(df_injected_model.loc[detected_mask, features])

In [ ]:
df_injected['penalty'] = np.where(df_injected['ensemble_score'] > 0.50, df_injected['ensemble_score'] * 20, 0)
station_health = 100 - df_injected.groupby('station_id')['penalty'].transform(lambda x: x.rolling(720, min_periods=1).sum()).clip(upper=100)
df_injected['sensor_health_score'] = station_health.clip(lower=0)

In [ ]:
def health_status(score):
    if score >= 85: return 'HEALTHY'
    elif score >= 60: return 'WARNING'
    else: return 'CRITICAL'

In [ ]:
df_injected['health_status'] = df_injected['sensor_health_score'].apply(health_status)

In [ ]:

df_injected

,cluster,station_id,station_name,city,latitude,longitude,timestamp,temperature_c,humidity_pct,pressure_hpa,...,pressure_hpa_roll6_var,temperature_c_zscore,humidity_pct_zscore,pressure_hpa_zscore,stat_zscore,ensemble_score,predicted_cause,penalty,sensor_health_score,health_status
24,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 00:00:00,6.0,100.0,1018.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.414831,Normal,0.000000,100.000000,HEALTHY
25,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 01:00:00,6.3,100.0,1018.4,...,0.080000,0.235702,0.000000,0.235702,0.157135,0.469230,Normal,0.000000,100.000000,HEALTHY
26,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 02:00:00,6.6,99.0,1019.1,...,0.310000,0.333333,0.384900,0.359211,0.359148,0.557604,Normal,11.152082,88.847918,HEALTHY
27,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 03:00:00,9.6,87.0,1020.2,...,0.929167,0.494580,0.498621,0.440902,0.478034,0.555264,Normal,11.105281,77.742637,WARNING
28,NCR,42139,Meerut,Meerut,29.0167,77.6333,2024-01-02 04:00:00,8.0,98.0,1020.7,...,1.327000,0.155902,0.072192,0.410896,0.212997,0.497824,Normal,0.000000,77.742637,WARNING
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459404,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 06:00:00,31.6,65.0,1008.9,...,0.221667,0.458007,0.539038,0.229093,0.408712,0.313368,Normal,0.000000,77.643426,WARNING
459405,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 07:00:00,32.0,63.0,1008.2,...,0.318667,0.528673,0.609002,0.054951,0.397542,0.299208,Normal,0.000000,77.643426,WARNING
459406,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 08:00:00,32.6,61.0,1006.9,...,1.165667,0.613659,0.650820,0.260904,0.508461,0.343651,Normal,0.000000,77.643426,WARNING
459407,Tamil_Nadu_Coast,43331,M.O. Pondicherry / Puducherry,Puducherry,11.9667,79.8167,2026-08-24 09:00:00,32.5,63.0,1005.6,...,2.582667,0.605287,0.573924,0.565930,0.581713,0.379111,Normal,0.000000,77.643426,WARNING


In [ ]:
df_injected.isna().sum()

,0
cluster,0
station_id,0
station_name,0
city,0
latitude,0
longitude,0
timestamp,0
temperature_c,0
humidity_pct,0
pressure_hpa,0


In [ ]:
df_clean.isna().sum()

,0
cluster,0
station_id,0
station_name,0
city,0
latitude,0
longitude,0
timestamp,0
temperature_c,0
humidity_pct,0
pressure_hpa,0


In [ ]:
import joblib
artifacts = {'iso_model': iso_model,'cause_classifier': cause_classifier,'optimal_threshold': best_thresh,'features': features}
joblib.dump(artifacts, 'skyguard_artifacts.pkl')
print("Artifacts saved successfully!")

Artifacts saved successfully!
